In [22]:
import pandas as pd
import numpy as np
import requests
from io import StringIO
import scipy

import os



# Datos Bucaramanga

In [2]:

# URL base de la API
base_url = "https://www.datos.gov.co/resource/57ur-5p28.csv"

# Parámetros
limit = 1000  # número máximo permitido por la API
offset = 0    # desplazamiento inicial
all_data = [] # lista para almacenar los bloques

csv_file = "..\\data\\datos_eh_bcmga_2021.csv"


In [ ]:
if os.path.exists(csv_file):
    df_bmga = pd.read_csv(csv_file)
    print("Se carga el archivo csv")
else:
    while True:
        # Construir URL con paginación
        url = f"{base_url}?$limit={limit}&$offset={offset}"
        print(f"Descargando: {url}")

        # Hacer la solicitud
        response = requests.get(url)
        
        # Si falla la solicitud, salir
        if response.status_code != 200:
            print(f"Error en la descarga: {response.status_code}")
            break

        # Leer el bloque como DataFrame
        df_chunk = pd.read_csv(StringIO(response.text))
        
        # Si el bloque está vacío, terminamos
        if df_chunk.empty:
            break
        
        # Agregar a la lista
        all_data.append(df_chunk)
        
        # Aumentar el offset para el siguiente bloque
        offset += limit
    # Unir todos los bloques en un solo DataFrame
    df_bmga = pd.concat(all_data, ignore_index=True)

    print(f"Total de registros descargados: {len(df_bmga)}")

    df_bmga.to_csv(csv_file, index=False)


Se carga el archivo csv


# Datos España

In [4]:
# Extract data from dataset

WS_data = scipy.io.loadmat('..\\data\\Weather_data.mat')
WS_data.keys()

dict_keys(['__header__', '__version__', '__globals__', 'Date', 'Lon', 'Lat', 'Alt', 'Temperature', 'WindSpeed', 'WindDirectionX', 'WindDirectionY', 'Pressure'])

In [5]:
columns = ['Date', 'Lon', 'Lat', 'Alt', 'Temperature', 
           'WindSpeed', 'WindDirectionX', 'WindDirectionY', 'Pressure']

In [6]:
# Crear el DataFrame
df_spna = pd.DataFrame({col: WS_data[col].flatten() for col in columns})

# Exploración de los datos

## Datos B/manga

In [7]:
df_bmga.head()

,estaci_n,tipo,coordenada_x,coordenada_y,date,time,out,hum,speed,dir,bar,rain,rad,index
0,ACAPULCO,CLIMATOLOGICA,1102787,1265648,2021-06-01T00:00:00.000,1899-12-31T00:00:00.000,22.1,98.0,0.0,SE,676.0,0.0,0.0,-999.0
1,ACAPULCO,CLIMATOLOGICA,1102787,1265648,2021-06-01T00:00:00.000,1899-12-31T01:00:00.000,20.3,96.0,0.4,SE,676.2,5.2,0.0,-999.0
2,ACAPULCO,CLIMATOLOGICA,1102787,1265648,2021-06-01T00:00:00.000,1899-12-31T02:00:00.000,20.1,99.0,0.4,WSW,675.8,25.6,0.0,-999.0
3,ACAPULCO,CLIMATOLOGICA,1102787,1265648,2021-06-01T00:00:00.000,1899-12-31T03:00:00.000,19.6,99.0,0.4,SSE,675.6,6.0,0.0,-999.0
4,ACAPULCO,CLIMATOLOGICA,1102787,1265648,2021-06-01T00:00:00.000,1899-12-31T04:00:00.000,19.8,99.0,0.0,SSE,675.9,0.2,0.0,-999.0


In [21]:
cardinal_to_deg = {
    "N": 0, "NNE": 22.5, "NE": 45, "ENE": 67.5,
    "E": 90, "ESE": 112.5, "SE": 135, "SSE": 157.5,
    "S": 180, "SSW": 202.5, "SW": 225, "WSW": 247.5,
    "W": 270, "WNW": 292.5, "NW": 315, "NNW": 337.5
}


In [30]:
# Ejemplo: tu DataFrame
df = df_bmga.copy()

# Normalizar a string para detectar cardinales
df['dir'] = df['dir'].astype(str).str.strip().str.upper()

# Convertir cardinales a grados
df['dir_deg'] = df['dir'].map(cardinal_to_deg)

# Si no es cardinal, intentar convertir a número
df['dir_deg'] = df['dir_deg'].fillna(pd.to_numeric(df['dir'], errors='coerce'))

# Reemplazar valores inválidos
df['dir_deg'] = df['dir_deg'].replace([-999, -999.0], np.nan)

# Si hay valores > 360, llevarlos al rango [0,360)
df['dir_deg'] = df['dir_deg'] % 360

print(df[['dir', 'dir_deg']].sample(20))


         dir  dir_deg
100344   NNE     22.5
89318      N      0.0
110953    38     38.0
26343    337    337.0
90271    ESE    112.5
124872    NW    315.0
12828    131    131.0
7924     WSW    247.5
85109    NNW    337.5
76897      W    270.0
107017   117    117.0
5826     ENE     67.5
30603    242    242.0
18401    160    160.0
129449  -999      NaN
23365    343    343.0
12899     47     47.0
31388    136    136.0
110430   126    126.0
54528     NW    315.0


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138566 entries, 0 to 138565
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   estaci_n      138566 non-null  object        
 1   tipo          138566 non-null  object        
 2   coordenada_x  138566 non-null  int64         
 3   coordenada_y  138566 non-null  int64         
 4   date          138566 non-null  datetime64[ns]
 5   time          138566 non-null  object        
 6   out           138566 non-null  float64       
 7   hum           138566 non-null  float64       
 8   speed         138566 non-null  float64       
 9   dir           138566 non-null  object        
 10  bar           138566 non-null  float64       
 11  rain          138566 non-null  float64       
 12  rad           138566 non-null  float64       
 13  index         138566 non-null  float64       
 14  datetime      138566 non-null  datetime64[ns]
 15  dir_deg       113

In [34]:
1 - 113480 / 138566 

0.18104008198259314

In [24]:
for val in df["dir"].unique(): print(val)

SE
WSW
SSE
ENE
NE
NNE
W
SW
NW
SSW
E
WNW
NNW
N
ESE
S
-999
187
154
235
211
144
217
39
51
228
240
221
254
177
130
147
163
91
128
215
68
111
94
216
243
279
231
185
210
157
156
109
198
281
132
124
1
92
321
74
42
113
87
167
146
117
78
344
218
263
170
271
136
237
67
61
98
183
212
5
247
232
176
102
259
63
70
181
265
209
286
261
152
159
93
172
193
60
50
104
47
44
205
35
54
223
36
359
266
161
65
116
140
149
182
101
62
82
184
227
129
200
169
141
58
119
137
229
244
250
249
138
219
304
121
122
64
174
201
225
84
48
108
339
248
224
234
222
206
126
305
253
145
270
125
252
40
213
142
204
112
274
29
202
81
241
90
208
230
292
327
347
153
268
4
30
143
9
164
238
337
134
194
196
77
27
318
195
155
162
110
99
239
226
53
115
186
57
86
158
83
107
131
276
88
72
175
85
207
71
31
103
242
349
192
69
258
188
284
148
135
251
55
127
179
191
133
100
80
300
96
151
75
56
24
180
324
52
354
272
168
246
306
257
287
214
89
114
199
178
348
11
139
233
203
285
79
95
23
245
260
314
262
173
256
59
160
277
190
355
197
32
12
334
33

In [9]:
# Supongamos que df_bmga tiene las columnas 'date' y 'time'
df_bmga['date'] = pd.to_datetime(df_bmga['date'])
df_bmga['time'] = pd.to_datetime(df_bmga['time']).dt.time  # extraer solo la hora

In [10]:
# Combinar
df_bmga['datetime'] = df_bmga.apply(lambda row: pd.Timestamp.combine(row['date'], row['time']), axis=1)

df_bmga[['date', 'time', 'datetime']].head()


,date,time,datetime
0,2021-06-01,00:00:00,2021-06-01 00:00:00
1,2021-06-01,01:00:00,2021-06-01 01:00:00
2,2021-06-01,02:00:00,2021-06-01 02:00:00
3,2021-06-01,03:00:00,2021-06-01 03:00:00
4,2021-06-01,04:00:00,2021-06-01 04:00:00


In [11]:
print(f"Número de estaciones: {df_bmga['estaci_n'].nunique()}")

Número de estaciones: 18


In [12]:
df_bmga.describe(include="object")

,estaci_n,tipo,time,dir
count,138566,138566,138566,138566
unique,18,1,24,1348
top,EL ROBLE,CLIMATOLOGICA,23:00:00,-999
freq,8760,138566,5775,24644


## Datos España

In [20]:
df_spna.sample()

,Date,Lon,Lat,Alt,Temperature,WindSpeed,WindDirectionX,WindDirectionY,Pressure
22933,[2018-03-05 03:20:00],5.255556,50.193611,294.0,4.5,4.0,0.322266,-0.946649,961.0


In [14]:
print(f"Número de estaciones: {df_spna['Lon'].nunique()}")

Número de estaciones: 21
